# Using MCP Server with LangChain Agents

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) lets you expose tools via a standard protocol that any MCP-compatible client can consume. This is especially powerful for **remote hosted MCP servers** — tools running as HTTP services that your agent can call over the network.

Microsoft exposes a public MCP server at `https://learn.microsoft.com/api/mcp` that provides tools to search and retrieve Microsoft documentation.

This is a great example of a **third-party hosted MCP server** — you don't need to deploy anything, just point your client at the URL.

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

- Completion of steps in `010_langchain_fundamentals` where we deployed an Azure Foundry with an LLM model.

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the endpoint and API key from the Terraform outputs. You can do this by running the following command.

In [1]:
foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name)

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4


Then we can test the endpoint and key by making a simple API call to the LLM model.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens= 512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- helping with coding, math, and problem-solving

A few useful things to know about me:
- I don’t have feelings, consciousness, or personal experiences.
- I generate responses based on patterns in data and the instructions I’m given.
- I can be very helpful, but I can also be wrong, so it’s good to verify important information.
- I don’t automatically know private or real-time information unless you provide it or my environment explicitly supplies it.

If you want, I can also tell you about:
- what I’m good at
- my limitations
- how to get better answers from me
- how I compare to a search engine or a human assistant

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the Microsoft Learn MCP server
mcp_client = MultiServerMCPClient(
    {
        "microsoft-learn": {
            "url": "https://learn.microsoft.com/api/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by Microsoft Learn
learn_tools = await mcp_client.get_tools()
print("Microsoft Learn MCP tools:", [t.name for t in learn_tools])

# Create an agent with Microsoft Learn tools MCP server
learn_agent = create_agent(model, learn_tools)

Microsoft Learn MCP tools: ['microsoft_docs_search', 'microsoft_code_sample_search', 'microsoft_docs_fetch']


### Query Microsoft Documentation

Ask the agent a question that requires looking up Microsoft documentation. The agent will call the Microsoft Learn MCP tools to search and retrieve relevant docs.

In [4]:
from langchain_core.messages import HumanMessage

async for step in learn_agent.astream(
    {"messages": [HumanMessage(content="How do I deploy a container app with GPU support on Azure Container Apps?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

How do I deploy a container app with GPU support on Azure Container Apps?
================================== Ai Message ==================================
Tool Calls:
  microsoft_docs_search (call_ZENonuwZbNgJyTsJlOaHw0Q5)
 Call ID: call_ZENonuwZbNgJyTsJlOaHw0Q5
  Args:
    query: Azure Container Apps GPU support deploy container app with GPU official docs
================================= Tool Message =================================
Name: microsoft_docs_search

[{'type': 'text', 'text': '{"results":[{"title":"Use GPUs with Azure Functions on Azure Container Apps","content":"# Use GPUs with Azure Functions on Azure Container Apps\\nIn this guide, you create and deploy a GPU-enabled function app on Azure Container Apps. You package your function code with GPU libraries into a container image, then deploy it to access NVIDIA T4 or A100 GPUs on demand.\\nBy combining Azure Functions\\u0027 serverless model